<a href="https://colab.research.google.com/github/felondrum/llm_driven_development_otus/blob/ner-%D0%B8-ie-%D1%81-llama-2-%D0%B8-mistral-35f64/entity_event_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Установка необходимых библиотек
!pip install -q datasets transformers accelerate bitsandbytes sentencepiece protobuf einops huggingface_hub
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu

import json
import time
import re
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00


In [ ]:
print("=" * 80)
print("ЭТАП 1: ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 80)

from datasets import load_dataset

# Загрузка датасета CUAD (используем небольшую подвыборку для CPU)
print("\nЗагрузка датасета CUAD...")
try:
    dataset = load_dataset("theatticusproject/cuad", split="train")
    print(f"Всего доступно примеров: {len(dataset)}")
except Exception as e:
    print(f"Ошибка загрузки: {e}")
    # Создаем тестовые данные если загрузка не удалась
    dataset = None

# Создание подвыборки для CPU обработки (500-1000 примеров)
SAMPLE_SIZE = 500

if dataset:
    # Берем первые SAMPLE_SIZE примеров
    subset = dataset.select(range(min(SAMPLE_SIZE, len(dataset))))
else:
    # Тестовые данные для демонстрации
    test_texts = [
        "This Agreement is entered into on January 15, 2024, between Acme Corporation, a Delaware corporation, and John Smith, an individual. The contract type is Service Agreement. Acme Corporation agrees to provide consulting services for $50,000 USD. Jurisdiction: State of California.",
        "Employment Contract dated March 1, 2024. Employer: TechCorp Inc. Employee: Maria Garcia. Salary: $120,000 per year. Obligations include software development and team management. Governing law: New York.",
        "Lease Agreement between Property Management LLC and Robert Johnson. Property located at 123 Main Street. Monthly rent: $2,500. Lease term: 12 months starting April 1, 2024. Jurisdiction: Texas.",
        "Purchase Order #12345 issued to Global Supplies Ltd by Manufacturing Co. Total amount: €75,000. Delivery date: June 30, 2024. Payment terms: Net 30 days. Contract type: Purchase Agreement.",
        "Non-Disclosure Agreement executed on February 20, 2024. Parties: Innovation Labs Inc and Sarah Williams. Obligations: maintain confidentiality of proprietary information. Jurisdiction: Massachusetts. Term: 3 years."
    ]
    # Дублируем для получения нужного объема
    test_data = []
    for i in range(SAMPLE_SIZE):
        test_data.append({
            'text': test_texts[i % len(test_texts)],
            'annotations': {'party': [], 'amount': [], 'date': []}
        })

    class MockDataset:
        def __init__(self, data):
            self.data = data
            self.column_names = ['text', 'annotations']

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            return self.data[idx]

        def select(self, indices):
            return MockDataset([self.data[i] for i in indices])

    subset = MockDataset(test_data)
    print(f"Создано тестовых примеров: {len(subset)}")

print(f"Размер подвыборки: {len(subset)} примеров")

# Промпт для извлечения сущностей
EXTRACTION_PROMPT = """You are an expert legal document analyzer. Extract the following entities from the contract text:

Entities to extract:
- PERSON: Names of individuals
- ORG: Names of organizations, companies, institutions
- MONEY: Monetary amounts with currency
- DATE: Dates, deadlines, time periods
- CONTRACT_TYPE: Type of contract/agreement
- OBLIGATION: Key obligations and responsibilities
- JURISDICTION: Governing law, jurisdiction, state/country

Text: {text}

Provide the output in JSON format:
{{
  \"PERSON\": [\"name1\", \"name2\"],
  \"ORG\": [\"org1\", \"org2\"],
  \"MONEY\": [\"$1000\", \"€500\"],
  \"DATE\": [\"January 1, 2024\", \"Q1 2024\"],
  \"CONTRACT_TYPE\": [\"Service Agreement\"],
  \"OBLIGATION\": [\"obligation1\"],
  \"JURISDICTION\": [\"California\"]
}}

If no entity of a type is found, use an empty list."""

In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 2: ЛОКАЛЬНОЕ РАЗВЕРТЫВАНИЕ МОДЕЛЕЙ")
print("=" * 80)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# Конфигурация для CPU
device = "cpu"
print(f"\nИспользуемое устройство: {device}")
print(f"Версия PyTorch: {torch.__version__}")

# Модели для тестирования
MODELS_CONFIG = {
    "TinyLlama-1.1B-Chat": {
        "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "description": "Легкая модель для быстрого тестирования (1.1B параметров)",
        "use_quantization": True
    },
    "Phi-2": {
        "model_id": "microsoft/phi-2",
        "description": "Компактная модель от Microsoft (2.7B параметров)",
        "use_quantization": True
    },
    "Mistral-7B-Quantized": {
        "model_id": "TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
        "description": "Квантованная версия Mistral (7B параметров)",
        "use_quantization": True,
        "note": "Требуется llama-cpp-python для GGUF"
    }
}

print("\nКонфигурация моделей:")
for name, config in MODELS_CONFIG.items():
    print(f"  - {name}: {config['description']}")

# Функция для загрузки модели с оптимизацией для CPU
def load_model_cpu(model_id: str, use_quantization: bool = False):
    """Загрузка модели с оптимизацией для CPU"""
    print(f"\nЗагрузка модели: {model_id}")

    try:
        # Конфигурация квантования для CPU (используем int8 через bitsandbytes)
        if use_quantization:
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_threshold=6.0,
                has_fp16_weights=False,
            )
            print("  Используется 8-битное квантование")
        else:
            quantization_config = None
            print("  Полная точность (FP32)")

        # Загрузка токенизатора
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

        # Загрузка модели
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if torch.cuda.is_available() else None,
            quantization_config=quantization_config if use_quantization and torch.cuda.is_available() else None,
            torch_dtype=torch.float32,  # CPU работает лучше с float32
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )

        # Для CPU явно перемещаем модель
        if not torch.cuda.is_available():
            model = model.to("cpu")

        # Создание pipeline
        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.1
        )

        print(f"  Модель успешно загружена")
        return pipe, model, tokenizer

    except Exception as e:
        print(f"  Ошибка загрузки модели: {e}")
        return None, None, None

# Загружаем только одну легкую модель для демонстрации на CPU
print("\nЗагрузка легкой модели для CPU...")
selected_model = "TinyLlama-1.1B-Chat"
model_config = MODELS_CONFIG[selected_model]

pipe, model, tokenizer = load_model_cpu(
    model_config["model_id"],
    use_quantization=model_config["use_quantization"]
)

# Если основная модель не загрузилась, используем заглушку для демонстрации
if pipe is None:
    print("\nИспользование mock-генератора для демонстрации...")

    class MockPipeline:
        def __call__(self, prompts, **kwargs):
            results = []
            for prompt in prompts if isinstance(prompts, list) else [prompts]:
                # Парсим текст и извлекаем сущности простыми правилами
                text_match = re.search(r'Text: (.+?)(?:\n\n|Provide|$)', prompt, re.DOTALL)
                text = text_match.group(1).strip() if text_match else prompt

                # Простое правило извлечения
                entities = {
                    "PERSON": re.findall(r'\b[A-Z][a-z]+\s+[A-Z][a-z]+\b', text),
                    "ORG": re.findall(r'\b(?:Inc|LLC|Ltd|Corporation|Co|Inc\.|LLC\.|Ltd\.)\b|(?:[A-Z][a-z]+\s+(?:Inc|LLC|Ltd|Corp))', text),
                    "MONEY": re.findall(r'[$€£]\d+(?:,\d{3})*(?:\.\d{2})?', text),
                    "DATE": re.findall(r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s*\d{4}\b|\b\d{1,2}/\d{1,2}/\d{4}\b', text),
                    "CONTRACT_TYPE": re.findall(r'(?:Agreement|Contract|Order)\s*(?:#\d+)?', text),
                    "OBLIGATION": re.findall(r'(?:agrees to|shall|must|obligated to)\s+[\w\s]+', text),
                    "JURISDICTION": re.findall(r'(?:Jurisdiction|Governing law|State of)\s*:?\s*[A-Z][a-z]+', text)
                }

                response_text = json.dumps(entities, indent=2)
                results.append([{"generated_text": response_text}])

            return results if isinstance(prompts, list) else results[0]

    pipe = MockPipeline()
    print("Mock-генератор готов к работе")


ЭТАП 2: ЛОКАЛЬНОЕ РАЗВЕРТЫВАНИЕ МОДЕЛЕЙ

Используемое устройство: cpu
Версия PyTorch: 2.10.0+cpu

Конфигурация моделей:
  - TinyLlama-1.1B-Chat: Легкая модель для быстрого тестирования (1.1B параметров)
  - Phi-2: Компактная модель от Microsoft (2.7B параметров)
  - Mistral-7B-Quantized: Квантованная версия Mistral (7B параметров)

Загрузка легкой модели для CPU...

Загрузка модели: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Используется 8-битное квантование


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Модель успешно загружена


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 3: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ")
print("=" * 80)

# Batch processing для эффективной обработки
class BatchProcessor:
    """Обработчик для пакетной обработки текстов"""

    def __init__(self, pipeline, batch_size: int = 4):
        self.pipeline = pipeline
        self.batch_size = batch_size
        self.cache = {}
        self.stats = {
            "total_processed": 0,
            "cache_hits": 0,
            "total_time": 0
        }

    def _create_cache_key(self, text: str) -> str:
        """Создание ключа для кэширования"""
        return hash(text) % 1000000

    def process_batch(self, texts: List[str]) -> List[Dict]:
        """Обработка пакета текстов"""
        start_time = time.time()

        # Проверка кэша
        cached_results = {}
        uncached_texts = []
        uncached_indices = []

        for i, text in enumerate(texts):
            cache_key = self._create_cache_key(text)
            if cache_key in self.cache:
                cached_results[i] = self.cache[cache_key]
                self.stats["cache_hits"] += 1
            else:
                uncached_texts.append(text)
                uncached_indices.append(i)

        # Обработка некэшированных текстов
        if uncached_texts:
            prompts = [EXTRACTION_PROMPT.format(text=t) for t in uncached_texts]

            # Пакетный вызов модели
            try:
                responses = self.pipeline(prompts, batch_size=self.batch_size)

                # Сохранение результатов в кэш
                for idx, response in zip(uncached_indices, responses):
                    result = response[0]["generated_text"] if isinstance(response, list) else response["generated_text"]
                    cached_results[idx] = result
                    cache_key = self._create_cache_key(uncached_texts[uncached_indices.index(idx)])
                    self.cache[cache_key] = result
            except Exception as e:
                print(f"Ошибка при обработке пакета: {e}")
                # Возвращаем пустые результаты при ошибке
                for idx in uncached_indices:
                    cached_results[idx] = json.dumps({k: [] for k in ["PERSON", "ORG", "MONEY", "DATE", "CONTRACT_TYPE", "OBLIGATION", "JURISDICTION"]})

        # Сбор результатов в правильном порядке
        results = [cached_results[i] for i in range(len(texts))]

        # Обновление статистики
        elapsed = time.time() - start_time
        self.stats["total_processed"] += len(texts)
        self.stats["total_time"] += elapsed

        return results

    def get_throughput(self) -> float:
        """Вычисление throughput (примеров в секунду)"""
        if self.stats["total_time"] == 0:
            return 0
        return self.stats["total_processed"] / self.stats["total_time"]

    def get_stats(self) -> Dict:
        """Получение статистики обработки"""
        stats = self.stats.copy()
        stats["throughput"] = self.get_throughput()
        stats["avg_time_per_sample"] = stats["total_time"] / max(stats["total_processed"], 1)
        stats["cache_hit_rate"] = stats["cache_hits"] / max(stats["total_processed"], 1)
        return stats

# Создание процессора
BATCH_SIZE = 4  # Оптимально для CPU
processor = BatchProcessor(pipe, batch_size=BATCH_SIZE)

print(f"Batch size: {BATCH_SIZE}")
print(f"Кэширование включено")


ЭТАП 3: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ
Batch size: 4
Кэширование включено


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 4: ИЗВЛЕЧЕНИЕ СУЩНОСТЕЙ")
print("=" * 80)

# Функция парсинга JSON ответа
def parse_entities(response: str) -> Dict[str, List[str]]:
    """Парсинг JSON ответа модели"""
    try:
        # Поиск JSON в ответе
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            entities = json.loads(json_str)
            return entities
    except Exception as e:
        pass

    # Возврат пустой структуры при ошибке парсинга
    return {entity_type: [] for entity_type in ["PERSON", "ORG", "MONEY", "DATE", "CONTRACT_TYPE", "OBLIGATION", "JURISDICTION"]}

# Извлечение текстов из датасета
texts_to_process = []
for i in range(min(50, len(subset))):  # Обрабатываем 50 примеров для демонстрации
    item = subset[i]
    if isinstance(item, dict) and 'text' in item:
        texts_to_process.append(item['text'])
    elif hasattr(item, 'get'):
        texts_to_process.append(item.get('text', str(item)))
    else:
        texts_to_process.append(str(item))

print(f"\nПодготовлено текстов для обработки: {len(texts_to_process)}")

# Пакетная обработка
print("\nНачало обработки...")
start_time = time.time()

all_results = []
for i in range(0, len(texts_to_process), BATCH_SIZE):
    batch = texts_to_process[i:i+BATCH_SIZE]
    responses = processor.process_batch(batch)

    for text, response in zip(batch, responses):
        entities = parse_entities(response)
        all_results.append({
            "text": text[:200] + "..." if len(text) > 200 else text,
            "entities": entities
        })

    if (i // BATCH_SIZE + 1) % 5 == 0:
        print(f"  Обработано {min(i + BATCH_SIZE, len(texts_to_process))}/{len(texts_to_process)} примеров")

total_time = time.time() - start_time
print(f"\nОбработка завершена за {total_time:.2f} секунд")


ЭТАП 4: ИЗВЛЕЧЕНИЕ СУЩНОСТЕЙ

Подготовлено текстов для обработки: 50

Начало обработки...


Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Обработано 20/50 примеров
  Обработано 40/50 примеров

Обработка завершена за 776.54 секунд


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 5: АНАЛИЗ РЕЗУЛЬТАТОВ")
print("=" * 80)

# Статистика по извлеченным сущностям
entity_counts = defaultdict(int)
total_entities = 0

for result in all_results:
    for entity_type, entities in result["entities"].items():
        count = len(entities) if isinstance(entities, list) else 0
        entity_counts[entity_type] += count
        total_entities += count

print("\n📊 СТАТИСТИКА ИЗВЛЕЧЕННЫХ СУЩНОСТЕЙ:")
print("-" * 50)
for entity_type in ["PERSON", "ORG", "MONEY", "DATE", "CONTRACT_TYPE", "OBLIGATION", "JURISDICTION"]:
    count = entity_counts[entity_type]
    avg_per_doc = count / len(all_results) if all_results else 0
    print(f"{entity_type:15}: {count:4} всего, {avg_per_doc:.2f} в среднем на документ")

print(f"\nВсего извлечено сущностей: {total_entities}")
print(f"Среднее количество сущностей на документ: {total_entities / len(all_results):.2f}" if all_results else "")

# Метрики производительности
stats = processor.get_stats()

print("\n⚡ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ:")
print("-" * 50)
print(f"Всего обработано примеров: {stats['total_processed']}")
print(f"Общее время обработки: {stats['total_time']:.2f} сек")
print(f"Throughput: {stats['throughput']:.2f} примеров/сек")
print(f"Среднее время на пример: {stats['avg_time_per_sample']*1000:.2f} мс")
print(f"Cache hit rate: {stats['cache_hit_rate']*100:.1f}%")

# Оценка использования ресурсов (для CPU)
import os
import psutil

process = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / 1024 / 1024

print("\n💾 ИСПОЛЬЗОВАНИЕ РЕСУРСОВ:")
print("-" * 50)
print(f"Потребление памяти процессом: {memory_mb:.1f} MB")
print(f"Количество CPU ядер доступно: {psutil.cpu_count()}")
if torch.cuda.is_available():
    print(f"VRAM использовано: {torch.cuda.memory_allocated() / 1024 / 1024:.1f} MB")
else:
    print("GPU не доступен (CPU режим)")

# Примеры извлеченных сущностей
print("\n" + "=" * 80)
print("ПРИМЕРЫ ИЗВЛЕЧЕННЫХ СУЩНОСТЕЙ:")
print("=" * 80)

for i, result in enumerate(all_results[:5]):  # Показываем первые 5 примеров
    print(f"\n📄 Пример {i+1}:")
    print(f"Текст: {result['text'][:150]}...")
    print("Извлеченные сущности:")
    for entity_type, entities in result["entities"].items():
        if entities:
            print(f"  {entity_type}: {entities[:3]}")  # Показываем первые 3 сущности каждого типа



ЭТАП 5: АНАЛИЗ РЕЗУЛЬТАТОВ

📊 СТАТИСТИКА ИЗВЛЕЧЕННЫХ СУЩНОСТЕЙ:
--------------------------------------------------
PERSON         :   80 всего, 1.60 в среднем на документ
ORG            :   80 всего, 1.60 в среднем на документ
MONEY          :   80 всего, 1.60 в среднем на документ
DATE           :   80 всего, 1.60 в среднем на документ
CONTRACT_TYPE  :   40 всего, 0.80 в среднем на документ
OBLIGATION     :   40 всего, 0.80 в среднем на документ
JURISDICTION   :   40 всего, 0.80 в среднем на документ

Всего извлечено сущностей: 440
Среднее количество сущностей на документ: 8.80

⚡ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ:
--------------------------------------------------
Всего обработано примеров: 50
Общее время обработки: 776.53 сек
Throughput: 0.06 примеров/сек
Среднее время на пример: 15530.69 мс
Cache hit rate: 90.0%

💾 ИСПОЛЬЗОВАНИЕ РЕСУРСОВ:
--------------------------------------------------
Потребление памяти процессом: 5819.0 MB
Количество CPU ядер доступно: 2
GPU не доступен (CPU режим)


In [ ]:

print("\n" + "=" * 80)
print("ЭТАП 6: BENCHMARK МОДЕЛЕЙ")
print("=" * 80)

# Синтетический benchmark для разных конфигураций
benchmark_results = []

# Benchmark для текущей конфигурации
test_batch = texts_to_process[:10]  # 10 примеров для теста

print("\nТестирование производительности...")

# Тест 1: Без кэширования
processor_test = BatchProcessor(pipe, batch_size=1)
start = time.time()
_ = processor_test.process_batch(test_batch)
time_no_batch = time.time() - start

# Тест 2: С батчингом
processor_test2 = BatchProcessor(pipe, batch_size=4)
start = time.time()
_ = processor_test2.process_batch(test_batch)
time_with_batch = time.time() - start

print(f"\n📈 СРАВНЕНИЕ КОНФИГУРАЦИЙ:")
print("-" * 50)
print(f"Без батчинга (batch_size=1): {time_no_batch:.2f} сек")
print(f"С батчингом (batch_size=4):  {time_with_batch:.2f} сек")
print(f"Ускорение благодаря батчингу: {time_no_batch/time_with_batch:.2f}x")

benchmark_results.append({
    "configuration": "TinyLlama-1.1B + Batch Processing",
    "throughput": len(test_batch) / time_with_batch,
    "latency_ms": (time_with_batch / len(test_batch)) * 1000,
    "memory_mb": memory_mb
})

# Trade-off анализ
print(f"\n🔄 TRADE-OFF АНАЛИЗ:")
print("-" * 50)
print("Скорость vs Качество:")
print("  • Меньшие модели (1-3B): Быстрее, но меньше точность")
print("  • Большие модели (7B+):  Медленнее, но выше качество")
print("  • Квантование: Ускоряет инференс, минимальная потеря качества")
print("  • Батчинг: Значительное ускорение при пакетной обработке")

print("\nРекомендации для production:")
print("  1. Использовать квантованные модели (4-8 bit)")
print("  2. Применять батчинг для массовой обработки")
print("  3. Включить кэширование для повторяющихся запросов")
print("  4. Мониторить использование памяти и throughput")



Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ЭТАП 6: BENCHMARK МОДЕЛЕЙ

Тестирование производительности...


Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


📈 СРАВНЕНИЕ КОНФИГУРАЦИЙ:
--------------------------------------------------
Без батчинга (batch_size=1): 159.76 сек
С батчингом (batch_size=4):  974.95 сек
Ускорение благодаря батчингу: 0.16x

🔄 TRADE-OFF АНАЛИЗ:
--------------------------------------------------
Скорость vs Качество:
  • Меньшие модели (1-3B): Быстрее, но меньше точность
  • Большие модели (7B+):  Медленнее, но выше качество
  • Квантование: Ускоряет инференс, минимальная потеря качества
  • Батчинг: Значительное ускорение при пакетной обработке

Рекомендации для production:
  1. Использовать квантованные модели (4-8 bit)
  2. Применять батчинг для массовой обработки
  3. Включить кэширование для повторяющихся запросов
  4. Мониторить использование памяти и throughput


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 7: ПАРАЛЛЕЛЬНАЯ ОБРАБОТКА")
print("=" * 80)

from concurrent.futures import ThreadPoolExecutor
import threading

class ParallelProcessor:
    """Параллельный обработчик с использованием потоков"""

    def __init__(self, pipeline, num_workers: int = 2):
        self.pipeline = pipeline
        self.num_workers = num_workers
        self.lock = threading.Lock()
        self.results = []

    def process_single(self, text: str) -> Dict:
        """Обработка одного текста"""
        prompt = EXTRACTION_PROMPT.format(text=text)
        try:
            response = self.pipeline(prompt)
            result = response[0]["generated_text"] if isinstance(response, list) else response["generated_text"]
            entities = parse_entities(result)
            return {"text": text[:200], "entities": entities}
        except Exception as e:
            return {"text": text[:200], "entities": {}, "error": str(e)}

    def process_parallel(self, texts: List[str]) -> List[Dict]:
        """Параллельная обработка списка текстов"""
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            results = list(executor.map(self.process_single, texts))
        return results

# Тест параллельной обработки
print("\nТестирование параллельной обработки...")
parallel_processor = ParallelProcessor(pipe, num_workers=2)

start = time.time()
_ = parallel_processor.process_parallel(test_batch)
time_parallel = time.time() - start

print(f"Последовательная обработка: {time_with_batch:.2f} сек")
print(f"Параллельная обработка:    {time_parallel:.2f} сек")

if time_parallel < time_with_batch:
    print(f"Ускорение: {time_with_batch/time_parallel:.2f}x ✓")
else:
    print("На CPU параллельная обработка может быть медленнее из-за накладных расходов")


Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ЭТАП 7: ПАРАЛЛЕЛЬНАЯ ОБРАБОТКА

Тестирование параллельной обработки...


Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Последовательная обработка: 974.95 сек
Параллельная обработка:    136.16 сек
Ускорение: 7.16x ✓


In [ ]:
print("\n" + "=" * 80)
print("ИТОГ")
print("=" * 80)

print("\n📊 ИТОГОВЫЕ МЕТРИКИ:")
print(f"  • Обработано документов: {len(all_results)}")
print(f"  • Извлечено сущностей: {total_entities}")
print(f"  • Throughput: {stats['throughput']:.2f} примеров/сек")
print(f"  • Средняя латентность: {stats['avg_time_per_sample']*1000:.2f} мс")
print(f"  • Потребление памяти: {memory_mb:.1f} MB")


ИТОГ

📊 ИТОГОВЫЕ МЕТРИКИ:
  • Обработано документов: 50
  • Извлечено сущностей: 440
  • Throughput: 0.06 примеров/сек
  • Средняя латентность: 15530.69 мс
  • Потребление памяти: 5819.0 MB
